<a href="https://colab.research.google.com/github/annasehgal/ai-studio-agentic-coding-workshop/blob/dev/agentic_coding_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Break-Through-Tech/ai-studio-agentic-coding-workshop/blob/main/agentic_coding_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic Coding Workshop: Predicting Hotel Cancellations

Gemini writes the code. Your team directs it and checks its work.

- Type prompts into Gemini (the spark icon at the bottom of the screen). Read every plan before it runs.
- Click into the TODO cell before you prompt Gemini, so its code lands in the right spot.
- Run each **CHECK** cell when a part is done.
- Don't let Gemini edit the SETUP or CHECK cells.
- The CHECK cells use a helper file, `workshop_checks.py`, that the SETUP cell loads for you. You don't need to open it. All you need to know is that it has functions that check the work Gemini produces.

*Data: a sample of the Hotel Booking Demand dataset (Antonio, Almeida and Nunes, Data in Brief, 2019), CC BY 4.0, with changes for teaching.*

In [2]:
# SETUP: run this cell first. It loads the data and the helper file behind the CHECK cells.
# Don't ask Gemini to change it.
!wget -q -O workshop_checks.py https://raw.githubusercontent.com/Break-Through-Tech/ai-studio-agentic-coding-workshop/main/workshop_checks.py
import workshop_checks as wc

raw_df = wc.load_data()
raw_df.head()

Loaded 20,000 bookings with 32 columns.


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,203,2016,December,49,2,2,5,2,...,No Deposit,250.0,NaN,0,Transient,66.8,0,0,Check-Out,2016-12-09
1,City Hotel,1,82,2015,July,29,16,0,3,2,...,No Deposit,9.0,NaN,0,Transient,76.5,0,0,Canceled,2015-07-16
2,City Hotel,0,25,2016,December,53,27,0,3,3,...,No Deposit,220.0,NaN,0,Transient-Party,60.0,0,1,Check-Out,2016-12-30
3,City Hotel,0,1,2016,March,11,9,0,1,1,...,No Deposit,9.0,NaN,0,Transient-Party,95.0,0,0,Check-Out,2016-03-10
4,City Hotel,0,1680,2017,April,16,16,2,2,2,...,No Deposit,9.0,NaN,0,Transient,108.0,0,0,Check-Out,2017-04-20


---
## Part 1: Explore

Paste this into Gemini:
> Look at `raw_df`. Give us a data quality report: missing values, duplicate rows, odd values, and columns stored as the wrong type. Don't change or clean anything yet.

Then run CHECK 1. What did Gemini catch? What did it miss?

In [3]:
import pandas as pd
import workshop_checks as wc

# Automatically load raw_df if it hasn't been loaded in the current session
if 'raw_df' not in locals() and 'raw_df' not in globals():
    print("raw_df was not defined. Loading data via workshop_checks...")
    raw_df = wc.load_data()

# 1. Shape of raw_df
print(f"DataFrame Shape: {raw_df.shape}\n")

# 2. General Info & Data Types
print("--- Column Types and Non-Null Counts ---")
display(raw_df.info())

# 3. Missing Values
missing_vals = raw_df.isnull().sum()
missing_percent = 100 * raw_df.isnull().sum() / len(raw_df)
missing_table = pd.concat([missing_vals, missing_percent], axis=1, keys=['Missing Values', '% of Total'])
display(missing_table[missing_table['Missing Values'] > 0])

# 4. Duplicate Rows
duplicates = raw_df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates} ({100 * duplicates / len(raw_df):.2f}%)\n")

# 5. Numerical columns summary (to check for odd/anomalous values)
print("--- Numerical Columns Summary ---")
display(raw_df.describe())

# 6. Categorical columns unique values summary
print("--- Categorical Columns Summary ---")
display(raw_df.describe(include=['object', 'category']))


DataFrame Shape: (20000, 32)

--- Column Types and Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           20000 non-null  object 
 1   is_canceled                     20000 non-null  int64  
 2   lead_time                       20000 non-null  int64  
 3   arrival_date_year               20000 non-null  int64  
 4   arrival_date_month              20000 non-null  object 
 5   arrival_date_week_number        20000 non-null  int64  
 6   arrival_date_day_of_month       20000 non-null  int64  
 7   stays_in_weekend_nights         20000 non-null  int64  
 8   stays_in_week_nights            20000 non-null  int64  
 9   adults                          20000 non-null  int64  
 10  children                        20000 non-null  int64  
 11  babies                

None

,Missing Values,% of Total
country,103,0.515
agent,2775,13.875
company,18845,94.225



Number of duplicate rows: 3244 (16.22%)

--- Numerical Columns Summary ---


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,20000.000000,20000.00000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000,20000.000000,17225.000000,1155.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,0.376900,702.81060,2016.158800,27.140950,15.855900,0.933650,2.503350,1.858000,0.104550,0.007650,0.033000,0.08805,0.154100,0.220650,86.786531,186.619048,2.281300,101.535115,0.060250,0.570550
std,0.484622,1684.68941,0.708172,13.642382,8.782812,0.994785,1.899571,0.628694,0.400159,0.107666,0.178641,0.82288,1.570789,0.647445,110.535579,130.274355,17.448965,47.748787,0.239213,0.798533
min,0.000000,0.00000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000,9.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,26.00000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,9.000000,67.000000,0.000000,68.445000,0.000000,0.000000
50%,0.000000,104.00000,2016.000000,27.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,14.000000,174.000000,0.000000,94.500000,0.000000,0.000000
75%,1.000000,296.00000,2017.000000,38.000000,24.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,229.000000,263.500000,0.000000,125.400000,0.000000,1.000000
max,1.000000,15096.00000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,3.000000,9.000000,1.000000,26.00000,63.000000,17.000000,531.000000,525.000000,391.000000,451.500000,3.000000,5.000000


--- Categorical Columns Summary ---


,hotel,arrival_date_month,meal,country,market_segment,distribution_channel,reserved_room_type,assigned_room_type,deposit_type,customer_type,reservation_status,reservation_status_date
count,20000,20000,20000,19897,20000,20000,20000,20000,20000,20000,20000,20000
unique,2,12,5,120,7,5,9,11,3,4,3,875
top,City Hotel,August,BB,PRT,Online TA,TA/TO,A,A,No Deposit,Transient,Check-Out,2015-10-21
freq,13203,2300,15487,8238,9343,16321,14514,12472,17421,15066,12462,253


In [4]:
# CHECK 1
wc.check_quality_report(raw_df)

------------------------------------------------------------
CHECK 1: What's actually in this data?
------------------------------------------------------------
Exact duplicate rows: 3,244
Missing values:
  company: 18,845 (94%)
  agent: 2,775 (14%)
  country: 103 (1%)
Month is stored as text (for example 'July'), not a number.
Note: a missing 'agent' means the guest booked directly.
It is not an error, so dropping those rows throws away real bookings.

Compare this with the agent's report.
What did it catch? What did it miss? Did it list anything we didn't?
(This list isn't everything. There's more to find later.)


---
## Part 2: Plan, Build, Check

**Plan.** Use the prompt we built together, or this one. Add one thing you learned in Part 1 to Context.

> **Goal:** Predict `is_canceled` with a logistic regression model.
>
> **Context:** This is hotel booking data. A missing `agent` means the guest booked directly. *(Add what you learned in Part 1.)*
>
> **Rules:** Clean the data into a new DataFrame called `clean_df`, and don't modify `raw_df`. Keep the `arrival_date_year` and `hotel` columns. Tell us every time a step drops rows, and how many. Use an 80/20 split with `random_state=42`. Make `model` a scikit-learn Pipeline that includes its preprocessing, and save the test data as `X_test` and `y_test`. Put your code in the TODO cells or right below them. Don't change the SETUP or CHECK cells.
>
> **Success check:** We have `clean_df`, `model`, `X_test`, and `y_test`, and you've reported accuracy, precision, and recall.
>
> Show us a step-by-step plan first. Ask us questions if anything is unclear. Don't write any code until we approve the plan.

Read the plan as a team. Change anything you disagree with before you approve it.

**Build.** Ask for the cleaning first: *"Do the cleaning steps only. Then stop and show us how many rows each step removed."* Run CHECK 2. Then ask Gemini to build the model.

**Check.** Run CHECK 3. Write down one decision Gemini made that you changed.

In [5]:
import pandas as pd

# Create a copy to prevent modifying raw_df
clean_df = raw_df.copy()

print(f"Initial row count: {len(clean_df)}")

# 1. Handle Duplicates
pre_dup_count = len(clean_df)
clean_df = clean_df.drop_duplicates()
post_dup_count = len(clean_df)
duplicates_removed = pre_dup_count - post_dup_count
print(f"Dropped duplicates: {duplicates_removed} rows removed (Current count: {post_dup_count})")

# 2. Handle Missing Values
# agent: Missing means direct booking (set to 0)
clean_df['agent'] = clean_df['agent'].fillna(0.0)

# company: Missing means individual booking (set to 0)
clean_df['company'] = clean_df['company'].fillna(0.0)

# country: Fill missing with 'Unknown'
clean_df['country'] = clean_df['country'].fillna('Unknown')

print("Missing values handled (0 rows dropped).")

# 3. Convert arrival_date_month to numbers
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
clean_df['arrival_date_month'] = clean_df['arrival_date_month'].map(month_map)

print("Month column converted from string to numerical indices (1-12).")
print(f"Final clean_df shape: {clean_df.shape}")


Initial row count: 20000
Dropped duplicates: 3244 rows removed (Current count: 16756)
Missing values handled (0 rows dropped).
Month column converted from string to numerical indices (1-12).
Final clean_df shape: (16756, 32)


In [ ]:
# CHECK 2: after cleaning
wc.check_cleaned(raw_df, clean_df)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score

# 1. Separate features and target
# We must drop target-leaking features: 'reservation_status' and 'reservation_status_date'
target = 'is_canceled'
leakage_features = ['reservation_status', 'reservation_status_date']
features = [col for col in clean_df.columns if col != target and col not in leakage_features]

X = clean_df[features]
y = clean_df[target]

# Identify numerical and categorical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Create preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

# 4. Define final model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 5. Fit the model
model.fit(X_train, y_train)

# 6. Evaluate and print metrics
y_pred = model.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))


Accuracy:  0.7897
Precision: 0.7036
Recall:    0.5395

              precision    recall  f1-score   support

           0       0.82      0.90      0.86      2327
           1       0.70      0.54      0.61      1025

    accuracy                           0.79      3352
   macro avg       0.76      0.72      0.73      3352
weighted avg       0.78      0.79      0.78      3352



In [ ]:
# CHECK 3: after training
wc.check_model(model, X_test, y_test)

**Red flag on CHECK 3?** Ask Gemini to walk through the features the model uses and find the problem. Fix it, retrain, and run CHECK 3 again. Why is the lower score the better model?

---
## Part 3: Find It Yourself

A hotel wants to predict *next* year's cancellations. CHECK 4 trains on 2015 and 2016 bookings and tests on 2017.

If something's wrong, the agent probably won't spot it. Use Gemini to explore, but you decide what's wrong and how to fix it. Then run CHECK 4 again.

In [ ]:
# CHECK 4
wc.check_future(clean_df)

In [ ]:
# TODO: explore the data and fix clean_df here. Then run CHECK 4 again.

**What did you find?**
- What was wrong:
- How you found it:
- Why didn't the agent catch it?

---
## Challenge

Build the model a real team would ship. Write your own prompts using the framework.

**Rules**
1. Fix everything from Parts 2 and 3 first.
2. `train_df` is 2015 and 2016 bookings. `test_df` is 2017 bookings.
3. Build one scikit-learn `Pipeline`, fit on a DataFrame, called `challenge_pipeline`.
4. Compare at least two model types with cross-validation on `train_df`.
5. Catch at least 75% of cancellations (recall of 0.75 or higher). Pick your threshold using `train_df` only.
6. Your score is precision on 2017 bookings. Higher is better.

In [ ]:
# TODO: create train_df, test_df, and challenge_pipeline.
# Then set chosen_threshold in the check cell below.

In [ ]:
# CHALLENGE CHECK: set your threshold, then run.
chosen_threshold = 0.5
score = wc.check_challenge(challenge_pipeline, train_df, test_df, threshold=chosen_threshold)

---
## More to Try
- **Team rules:** add Custom Instructions with your team's norms, then rerun a task and compare.
- **Your project:** open your Challenge Project notebook and start with an Explore prompt.